# Xây dựng 7 nhóm nhân tố Alpha

<!--
Task: Mỗi nhóm sẽ nghiên cứu khoảng 3-5 feature và tạo template, viết từng class cho các feature.

1. Giá
2. Động lượng - vd: rsi
3. Overlap - vd: ma
4. Thống kê - vd: z-score
5. Xu hướng - vd: adx
6. Biến động - vd: atr
7. Volume - vd: volume
-->

## 1. Nhóm Giá / Mẫu hình (Price / Candlestick)
- Log Return.
- Heikin-Ashi (HA).
- Marubozu (Quantified).
- Hikkake.

##### 1.1 Log Return (Lợi suất Logarit)

###### A. Công thức Toán học:
- Simple Return:
$$Return_t = \dfrac{C_t - C_{t-1}}{C_{t-1}}$$
- Log Return:
$$Log\_return_t = ln\Big( \dfrac{C_t}{C_{t-1}} \Big) = ln(C_t) - ln(C_{t-1})$$
Khi biến động giá rất nhỏ, theo khai triển Taylor, ta có: $ln(1+x)\approx x$. Do đó:
$$ln\Big( \dfrac{C_t}{C_{t-1}} \Big) = ln(1+R_t) \approx Return_t$$

###### B. Template Code:

In [ ]:
import backtrader as bt
import math

class LogReturn(bt.Indicator):
    lines = ('log_return',)
    params = (('period', 1),)

    def __init__(self):
        self.addminperiod(self.p.period + 1)

    def next(self):
        current_close = self.data.close[0]
        past_close = self.data.close[-self.p.period]

        # log(Close_t / Close_t-n)
        self.lines.log_return[0] = math.log(current_close / past_close)

###### C. Ý nghĩa:
- Đo lường tốc độ thay đổi của giá. Thay vì tính bằng điểm số (ví dụ: tăng 5 điểm từ 1000 lên 1005 sẽ khác với tăng 5 điểm từ 1500 lên 1505), LogReturn chuẩn hóa mọi biến động về một thang đo duy nhất.

###### D. Cách dùng:
- Đo lường Độ biến động: Tính độ lệch chuẩn của LogReturn trong 20 nến gần nhất. Nếu Volatility đang cao đột biến, phải tự động giảm size lệnh (giảm weigh nếu nghiên cứu cổ phiếu) để bảo vệ vốn.

##### 1.2 Nến Heikin-Ashi (HA)

###### A. Công thức Toán học:
\begin{align*}
&HA\_Close_t = \dfrac{O_t + H_t + L_t + C_t}{4} \tag{1}
\\
&HA\_Open_t = \dfrac{HA\_Open_{t-1} + HA\_Close_{t-1} }{2} \tag{2}
\\
&HA\_High_t = \max{ (H_t , HA\_Open_t, HA\_Close_t) } \tag{3}
\\
&HA\_Low_t = \min{( L_t, HA\_Open_t, HA\_Close_t )} \tag{4}
\end{align*}

Phân tích phương trình (2), nhắc lại công thức EMA:
$$EMA_t = \alpha \cdot X_t + (1-\alpha) \cdot EMA_{t-1}$$
Thay $\alpha=0.5$ và $X_t=HA\_Close_{t-1}$, ta có:
$$EMA_t = 0.5 \cdot HA\_Close_{t-1} + 0.5 \cdot EMA_{t-1} = \dfrac{HA\_Close_{t-1} + EMA_{t-1} }{2}$$

$\Rightarrow$ $HA\_Open_t$ là $EMA$ của chuỗi $HA\_Close$ trễ 1 nến, với hệ số làm mượt $\alpha = 0.5$.

###### B. Template Code:

In [ ]:
import backtrader as bt
import math
class HeikinAshi(bt.Indicator):
    lines = ('ha_open', 'ha_high', 'ha_low', 'ha_close',)

    def next(self):
        O = self.data.open[0]
        H = self.data.high[0]
        L = self.data.low[0]
        C = self.data.close[0]

        # 1. HA Close
        ha_close = (O + H + L + C) / 4
        self.lines.ha_close[0] = ha_close

        # 2. HA Open
        if len(self) == 1:
            ha_open = (O + C) / 2
        else:
            # (HA_Open_t-1 + HA_Close_t-1) / 2
            ha_open = (self.lines.ha_open[-1] + self.lines.ha_close[-1]) / 2

        self.lines.ha_open[0] = ha_open

        # 3. HA High & HA Low
        self.lines.ha_high[0] = max(H, ha_open, ha_close)
        self.lines.ha_low[0] = min(L, ha_open, ha_close)

###### C. Ý nghĩa:
- Nến Nhật thông thường (OHLC) cung cấp bức tranh thô ráp. Khung 5 Phút (5M) của VN30F1M thường xuất hiện chuỗi nến Đỏ - Xanh - Đỏ - Xanh xen kẽ nhau liên tục (Chop market). Nến HA đóng vai trò là "Cặp kính râm", lọc bỏ những tia sáng chói (nhiễu), giúp bạn nhìn rõ xem Dòng chảy chính (Trend) đang là tăng hay giảm.

###### D. Cách dùng:
- Làm bộ lọc: Chỉ mở lệnh LONG khi nến HA màu xanh (HA_Close > HA_Open), và chỉ SHORT khi HA màu đỏ.
- Không dùng giá của Heikin-Ashi (HA_Close, HA_Open) để entry hay exit trong Backtest. Chỉ dùng HA làm signal, còn vào lệnh vẫn phải dùng giá thật của nến Nhật.

##### 1.3 Quantified Marubozu (Nến Marubozu)

###### A. Công thức Toán học:
$$M\_Ratio_t = \dfrac{Close_t - Open_t}{High_t - Low_t + 10^{-8}} \in [-1,1]$$

###### B. Template Code:

In [ ]:
import backtrader as bt
import math

class QuantifiedMarubozu(bt.Indicator):
    lines = ('marubozu_ratio',)

    def next(self):
        O = self.data.open[0]
        H = self.data.high[0]
        L = self.data.low[0]
        C = self.data.close[0]

        real_body = C - O
        high_low_range = H - L

        ratio = real_body / (high_low_range + 1e-8)

        self.lines.marubozu_ratio[0] = max(-1.0, min(1, ratio))

###### C. Ý nghĩa:
- Con số từ -1 đến 1 đo lường "độ hung hãn" của dòng tiền lớn (Cá mập). Một nến có Marubozu Ratio > 0.8 chứng tỏ phe mua/bán đã kiểm soát từ giây đầu tiên mở nến đến giây cuối cùng đóng nến mà không có sự phản kháng đáng kể nào từ phe đối lập.

###### D. Cách dùng:
- Xác nhận Breakout: Nếu giá vượt đường kháng cự + M_Ratio > 0.8 $\Rightarrow$ Vào lệnh.
- Trailing Stop: Nếu đang cầm lệnh Long, và tự nhiên xuất hiện một cây nến có M_Ratio < -0.85, đó là cảnh báo nên chốt lời ngay lập tức.

##### 1.4 Quantified Takuri (Nến rút chân Takuri)

###### A. Cơ sở lý thuyết:
Nến Takuri xuất hiện ở cuối một xu hướng giảm, thể hiện việc phe gấu (bán) đã cố gắng đạp giá xuống cực sâu, nhưng bị phe bò (mua) phản công và hấp thụ toàn bộ lực bán, đẩy giá đóng cửa về sát mức cao nhất phiên.
Một nến Takuri hoàn hảo có 3 tính chất:
- Lower Shadow (Bóng dưới) cực lớn.
- Real Body (Thân nến) cực nhỏ.
- Upper Shadow (Bóng trên) gần như bằng không.

###### B. Công thức Toán học:
- Biên độ nến: $$Range_t = H_t - L_t$$
- Thân nến: $$Body_t = |O_t - C_t|$$
- Bóng trên: $$US_t = H_t - \max{(O_t,C_t)}$$
- Bóng dưới: $$LS_t = \min{(O_t,C_t)} - L_t$$

Để đánh giá sức mạnh của phe mua (rút chân), ta lấy chiều dài bóng dưới trừ đi các thành phần "kém tích cực" (thân nến và bóng trên), sau đó chuẩn hóa trên tổng biên độ:
$$Takuri\_Score = \dfrac{LS_t - (Body_t + US_t)}{Range_t + 10^{-8}}$$

In [1]:
import backtrader as bt

class QuantifiedTakuri(bt.Indicator):
    lines = ('takuri_score',)


    def next(self):
        O = self.data.open[0]
        H = self.data.high[0]
        L = self.data.low[0]
        C = self.data.close[0]

        R = H - L
        RB = abs(O - C)
        US = H - max(O, C)
        LS = min(O, C) - L

        rejection_strength = LS - (RB + US)

        score = rejection_strength / (R + 1e-8)

        self.lines.takuri_score[0] = max(0, score)

###### D. Cách dùng:
- Mean Reversion: Tìm kiếm tín hiệu phân kỳ. Ví dụ: Giá VN30F1M chạm dải dưới của Bollinger Bands (hoặc đường Hỗ trợ cứng) và Takuri_Score > 0.7 thì có thể bắt đáy.
- Nếu đang giữ lệnh SHORT, mà thấy thị trường liên tục xuất hiện các nến có Takuri_Score > 0.5 (Giá cứ đạp xuống là bị múc lên), nên chuẩn bị thoát lệnh.


## 2. Nhóm Động lượng (Momentum)
- Fisher Transform (FISHER): Một phép biến đổi toán học phi tuyến tính tuyệt đẹp. Nó ép phân phối giá thành một phân phối chuẩn (Gaussian). Giúp các điểm đảo chiều trở nên sắc nét cực kỳ (sharp turning points).
- Schaff Trend Cycle (STC): Giải quyết độ trễ của MACD bằng cách cho nó đi qua thuật toán Stochastic 2 lần. Bắt chu kỳ (cycle) cực chuẩn.
- Qualitative Quantitative Estimation (QQE): Làm mượt RSI và kết hợp với Volatility trailing. Rất ít nhiễu (whipsaw).

##### 2.1 Fisher Transform (Biến đổi Fisher)

###### A. Công thức Toán học:
- Đầu tiên, ta chuẩn hóa giá $\Big($thường dùng giá Median: $HL2_t = \dfrac{H_t+L_t}{2}\Big)$ về khoảng $[-1,1]$ trong $N$ chu kỳ:
$$x_t = \dfrac{2 \times HL2_t - Min_N(HL2)}{Max_N(HL2) - Min_N(HL2)} - 1$$
- Sau đó, áp dụng phép biến đổi Fisher (bản chất là hàm arctanh - nghịch đảo của tang hyperbolic):
$$Fisher_t = \dfrac{1}{2} ln\Big( \dfrac{1+x_t}{1-x_t} \Big)$$

###### B. Template Code:

In [ ]:
import backtrader as bt
import math

class FisherTransform(bt.Indicator):
    lines = ('fisher', 'signal',)
    params = (('period', 9),)

    def __init__(self):
        hl2 = (self.data.high + self.data.low) / 2

        self.max_hl2 = bt.indicators.Highest(hl2, period=self.p.period)
        self.min_hl2 = bt.indicators.Lowest(hl2, period=self.p.period)
        self.prev_fisher = 0
        self.prev_x = 0

    def next(self):
        range_hl2 = self.max_hl2[0] - self.min_hl2[0]

        if range_hl2 == 0:
            x = 0
        else:
            hl2 = (self.data.high[0] + self.data.low[0]) / 2
            x = 2 * ((hl2 - self.min_hl2[0]) / range_hl2) - 1

        x = 0.5 * x + 0.5 * self.prev_x

        x = max(-0.999, min(0.999, x))

        fisher = 0.5 * math.log((1 + x) / (1 - x))
        fisher = 0.5 * fisher + 0.5 * self.prev_fisher

        self.lines.fisher[0] = fisher
        self.lines.signal[0] = self.prev_fisher

        self.prev_x = x
        self.prev_fisher = fisher

###### C. ý nghĩa:
- Nếu dùng RSI, giá có tăng mạnh cỡ nào RSI cũng chỉ lết từ 80 lên 85 rồi 90. Trong khi đó, Fisher Transform ép phân phối giá thành phân phối chuẩn. Về mặt xác suất, một sự kiện nằm cách mức trung bình quá 2 hoặc 3 độ lệch chuẩn là một sự kiện cực kỳ hiếm và có xác suất đảo chiều lên tới 95%. Fisher biến những dao động bình thường thành một đường đi ngang quanh mốc 0, nhưng khi giá bị đẩy tới ngưỡng "cực đoan", đường Fisher sẽ dựng đứng thẳng tắp lên các mốc +2, +3 hoặc cắm phập xuống -2, -3.

###### D. Cách dùng:
- Mean Reversion: Khi Fisher vượt quá +2 hoặc rớt dưới -2 và đường Fisher bắt đầu cắt xuống/lên đường Signal của chính nó $\Rightarrow$ Đây là thời điểm vào lệnh Đảo chiều với rủi ro thấp nhất.
- Chốt lời: Đang ôm Long ăn được 5 điểm, thấy Fisher dựng đứng lên +2.5, đóng vị thế ngay lập tức. Không tham lam gồng lãi khi xác suất toán học đã cạn kiệt.

##### 2.2 Schaff Trend Cycle (STC)

###### A. Cơ sở lý thuyết:
- MACD nhận diện xu hướng tốt nhưng rất trễ.
- Stochastic nhanh nhưng cực kỳ nhiễu (cắt lên cắt xuống liên tục).
Doug Schaff đã tạo ra STC bằng cách: Tính MACD $\to$ Đưa chuỗi MACD qua hàm Stochastic $\to$ Làm mượt $\to$ Đưa qua hàm Stochastic lần 2.

###### B. Công thức Toán học
- $$MACD_t = EMA(C,fast\_period) - EMA(C,slow\_period)$$
- $$(Stoch_1)_t = 100 \times \dfrac{MACD_t - Min_N(MACD)}{Max_n(MACD) - Min_N(MACD)} $$
- $$(Smoothed_1)_t = EMA\big( (Stoch_1)_t, smooth\_period \big)$$
- $$(Stoch_2)_t = 100 \times \dfrac{(Smoothed_1)_t - Min_N(Smoothed_1)}{Max_n(Smoothed_1) - Min_N(Smoothed_1)} $$
- $$STC = EMA\big( (Stoch_2)_t, smooth\_period \big)$$

###### C. Template Code:

In [ ]:
import backtrader as bt
import math

class SchaffTrendCycle(bt.Indicator):
    lines = ('stc',)
    params = (
        ('fast', 23), ('slow', 50),
        ('cycle', 10), ('smooth', 3)
    )

    def __init__(self):
        macd = bt.indicators.EMA(self.data, period=self.p.fast) - bt.indicators.EMA(self.data, period=self.p.slow)

        lowest_macd = bt.indicators.Lowest(macd, period=self.p.cycle)
        highest_macd = bt.indicators.Highest(macd, period=self.p.cycle)

        macd_range = highest_macd - lowest_macd
        stoch1 = 100 * bt.If(macd_range > 0, (macd - lowest_macd) / macd_range, 0)

        smooth1 = bt.indicators.EMA(stoch1, period=self.p.smooth)

        lowest_s1 = bt.indicators.Lowest(smooth1, period=self.p.cycle)
        highest_s1 = bt.indicators.Highest(smooth1, period=self.p.cycle)

        s1_range = highest_s1 - lowest_s1
        stoch2 = 100 * bt.If(s1_range > 0, (smooth1 - lowest_s1) / s1_range, 0)

        self.lines.stc = bt.indicators.EMA(stoch2, period=self.p.smooth)

###### D. Ý nghĩa:
- Khi có trend mạnh, STC sẽ dính chặt ở mức 100 (hoặc 0) thành một đường thẳng ngang, tuyệt đối không dao động nhiễu.

###### E. Cách dùng:
- Chống lệnh "Ngược Trend" (Filter): Nếu STC đang ở mức 100, tuyệt đối không mở lệnh SHORT dù cho chạm kháng cự hay RSI quá mua.
- Bắt nhịp Pullback: Giả sử Trend tổng đang là TĂNG. Đỏ vài cây nến. Đường STC rớt từ 100 về sát mốc 25 (vùng quá bán) và móc ngược lên $\Rightarrow$ Mở lệnh LONG. Đây là điểm vào lệnh hùa theo xu hướng an toàn nhất, ăn trọn con sóng đẩy thứ 2.

##### 2.3 Qualitative Quantitative Estimation (QQE)

###### A. Cơ sở lý thuyết:
- RSI thường đưa ra các tín hiệu nhiễu khi thị trường biến động cao.
QQE giải quyết điều này bằng cách áp dụng Trailing Stop dựa trên Volatility (giống kỹ thuật ATR) trực tiếp lên chỉ báo RSI thay vì áp lên giá.

###### B. Công thức Toán học:
- $$RSIndex_t = EMA\big( RSI(14) , 5 \big)$$
-  Tính biến động của RSI: $$\Delta_t = |RSIndex_t - RSIndex_{t-1}|$$
- Tính ATR của RSI: $$ATR\_RSI_t=EMA(\Delta_t,27)$$
- Làm mượt ATR_RSI: $$Smoothed\_ATR_t = EMA(ATR\_RSI_t,27)$$
- Tính dải Trailing Fast/Slow bằng cách nhân $Smoothed\_ATR_t$ với một hệ số (tùy chỉnh, do researcher làm).

###### C. Template Code:

In [ ]:
import backtrader as bt
import math

class QQE(bt.Indicator):
    lines = ('rsi_ema', 'fast_atr_rsi')
    params = (
        ('rsi_period', 14), ('rsi_smooth', 5),
        ('atr_period', 27), ('fast_mult', 4.236)
    )

    def __init__(self):
        # 1. Tính RSI và làm mượt
        rsi = bt.indicators.RSI(self.data, period=self.p.rsi_period)
        self.rsi_ema = bt.indicators.EMA(rsi, period=self.p.rsi_smooth)
        self.lines.rsi_ema = self.rsi_ema # Đường QQE chính

        self.addminperiod(self.p.atr_period * 2)

        self.prev_atr_rsi = 0
        self.prev_fast_line = 0

    def next(self):
        delta = abs(self.rsi_ema[0] - self.rsi_ema[-1])

        alpha = 2 / (self.p.atr_period + 1.0)

        # EMA của Delta
        if len(self) == self.p.atr_period * 2:
            self.prev_atr_rsi = delta
        else:
            self.prev_atr_rsi = alpha * delta + (1 - alpha) * self.prev_atr_rsi

        # EMA lần 2 của Delta
        smoothed_atr = self.prev_atr_rsi

        # 5. Tính Fast Trailing Line
        dar = self.prev_atr_rsi * self.p.fast_mult

        new_shortband = self.rsi_ema[0] + dar
        new_longband = self.rsi_ema[0] - dar

        prev_rsi = self.rsi_ema[-1]
        prev_fast = self.prev_fast_line

        if prev_rsi > prev_fast and self.rsi_ema[0] > prev_fast:
            fast_line = max(prev_fast, new_longband)
        elif prev_rsi < prev_fast and self.rsi_ema[0] < prev_fast:
            fast_line = min(prev_fast, new_shortband)
        else:
            fast_line = new_longband if self.rsi_ema[0] > prev_fast else new_shortband

        self.lines.fast_atr_rsi[0] = fast_line
        self.prev_fast_line = fast_line

###### D. Ý nghĩa:
- QQE lấy RSI làm mượt, sau đó xây dựng một đường Trailing Stop (Fast Line) bao quanh nó dựa trên độ biến động (ATR của chính RSI). Nếu giá chỉ giật nhẹ lên xuống, RSI có nhúc nhích nhưng không đủ sức xuyên thủng Fast Line $\Rightarrow$ Tín hiệu xu hướng không thay đổi.

###### E. Cách dùng:
- Sử dụng QQE trên khung thời gian lớn hơn 1 bậc (Ví dụ: Trade khung 1M thì nhìn QQE khung 5M). Nếu QQE khung 5M đang báo Long (RSI_EMA > Fast Line), mọi tín hiệu Short ở khung 1M đều bị bỏ qua.
- Trailing Stop: Trong một lệnh Long, giữ lệnh cho đến khi nào đường RSI_EMA cắt xuống dưới đường Fast Line.

## 3. Nhóm Overlap (Làm mượt / Lọc nhiễu)
- Arnaud Legoux Moving Average (ALMA): Dùng phân phối Gauss (Normal Distribution) để gán trọng số, giảm độ trễ (lag) đến mức tối thiểu mà vẫn mượt. Khắc phục hoàn toàn nhược điểm của SMA/EMA.
- Hull Moving Average (HMA): Sử dụng trung bình có trọng số của một nửa chu kỳ trừ đi chu kỳ đầy đủ. Tốc độ bám giá cực nhanh.
- Ehler's Super Smoother Filter (SSF): Áp dụng bộ lọc thông thấp (Low-pass filter) trong Xử lý tín hiệu số (DSP) để loại bỏ nhiễu tần số cao. Góc nhìn kỹ thuật thuần túy.

##### 3.1  Arnaud Legoux Moving Average (ALMA)

###### A. Cơ sở lý thuyết:
- Thay vì gán trọng số lớn nhất cho mức giá gần nhất (như EMA), ALMA sử dụng Phân phối Chuẩn để gán trọng số. Bằng cách di chuyển đỉnh của đường cong hình chuông này (thông qua tham số offset), ta có thể quyết định xem "sức nặng" của dữ liệu nằm ở hiện tại, hay lùi lại một chút trong quá khứ. Nó giống như việc chỉnh tiêu cự của máy ảnh để làm mờ các gai nhiễu hai bên.

###### B. Công thức Toán học:
Gọi $N$ là chu kỳ, $offset \in [0,1]$ (thường là $0.85$) là vị trí đỉnh chuông, $\sigma (thường là $6$) là độ thuôn của chuông.
- Tâm điểm: $$m=  \lfloor offset \times (N-1) \rfloor$$

trong đó: $\lfloor x \rfloor$ là số nguyên lớn nhất nhỏ hơn hoặc bằng $x$.

- Độ lệch chuẩn: $$s = \dfrac{N}{\sigma}$$
- Trọng số tại vị trí $i$ ($i\in \{ 0,\dots, N-1\}$):
$$W_i = exp\Big( - \dfrac{(i-m)^2}{2s^2} \Big)$$
-  Công thức ALMA: $$ALMA = \dfrac{\displaystyle\sum_{i=0}^{N-1} W_i \times C_{t-N + 1 + i}}{\displaystyle\sum_{i=0}^{N-1} W_i}$$

###### C. Template Code:

In [ ]:
import backtrader as bt
import math
import numpy as np

class ALMA(bt.Indicator):
    lines = ('alma',)
    params = (('period', 9), ('offset', 0.85), ('sigma', 6),)

    def __init__(self):
        self.addminperiod(self.p.period)

        m = math.floor(self.p.offset * (self.p.period - 1))
        s = self.p.period / self.p.sigma

        weights = np.zeros(self.p.period)
        for i in range(self.p.period):
            weights[i] = math.exp(-((i - m)**2) / (2 * s**2))

        self.weights = weights / np.sum(weights)

    def next(self):
        prices = self.data.get(size=self.p.period)

        self.lines.alma[0] = np.dot(prices, self.weights)

###### D. Ý nghĩa:
- ALMA loại bỏ hoàn toàn các râu nến quét thanh khoản nhanh. Nó mượt hơn SMA rất nhiều nhưng lại bám giá sát hơn.

###### E. Cách dùng:
- Làm Đường Hỗ trợ/Kháng cự động (Dynamic S/R). Trong khung 5M hoặc 10M, khi giá hồi về chạm đường ALMA(50) và bật lên, đó là điểm nhồi lệnh hoàn hảo. Nó tốt hơn EMA vì không bị gãy khi gặp một cây nến bất thường.

##### 3.2 Hull Moving Average (HMA)

###### A.  Cơ sở lý thuyết:
- Alan Hull đã phá vỡ giới hạn độ trễ bằng một suy luận toán học cực kỳ dị: Nếu tôi lấy giá trị trung bình của chu kỳ ngắn, nhân đôi nó lên, rồi trừ đi giá trị trung bình của chu kỳ dài, tôi sẽ tạo ra một đường chạy trước cả giá hiện tại (Overshoot/Lead).
Để triệt tiêu độ nhiễu của phép tính này, ông làm mượt kết quả bằng Căn bậc hai của chu kỳ.

##### B. Công thức Toán học:
- Công thức kết hợp sử dụng Đường trung bình có trọng số (WMA - Weighted Moving Average):
- $$Raw\_HMA_t = 2\times WMA\Big( C,\lfloor \dfrac{N}{2} \rfloor \Big)_t - WMA(C,N)_t $$
-  $$HMA = VWA(Raw\_HMA_t,\lfloor \sqrt{N} \rfloor)$$

###### C. Template Code:

In [ ]:
class HMA(bt.Indicator):
    """
    Hull Moving Average: Độ trễ tiệm cận 0, bám giá cực sát, phản ứng đảo chiều ngay lập tức.
    """
    lines = ('hma',)
    params = (('period', 14),)

    def __init__(self):
        # Tính chu kỳ theo công thức Hull
        half_period = int(self.p.period / 2)
        sqrt_period = int(math.sqrt(self.p.period))

        # 1. WMA nửa chu kỳ (nhân 2)
        wma_half = bt.indicators.WMA(self.data, period=half_period) * 2.0

        # 2. WMA toàn chu kỳ
        wma_full = bt.indicators.WMA(self.data, period=self.p.period)

        # 3. Raw HMA
        raw_hma = wma_half - wma_full

        # 4. Làm mượt lần cuối bằng WMA căn bậc 2 (Gắn thẳng vào line chính)
        self.lines.hma = bt.indicators.WMA(raw_hma, period=sqrt_period)

###### D. Ý nghĩa:
- HMA chạy cực kỳ sát giá. Khi giá đảo chiều, HMA sẽ chuyển hướng (đổi độ dốc) gần như cùng lúc với cây nến đảo chiều.

###### E. Cách dùng:
- Tín hiệu vào lệnh: Khi HMA chuyển từ dốc xuống sang dốc lên (Đạo hàm bậc 1 của HMA > 0) $\to$ Long ngay lập tức.
- Nhược điểm cần chú ý: Vì HMA có tính chất "chạy trước" (overshoot), nó đôi khi bị vọt lố qua giá thực. Không dùng HMA làm đường cắt lỗ, chỉ dùng để lấy tín hiệu.

##### 3.3 Ehler's Super Smoother Filter (SSF)

###### A. Công thức Toán học:
Công thức là một phương trình đệ quy vi phân:
- Tính các hằng số:
    + $$\alpha = exp\Big( \dfrac{-\sqrt{2}\pi}{N} \Big)$$
    + $$\beta = 2\alpha \cos{\Big( \dfrac{\sqrt{2}\pi}{N} \Big)}$$
    + $$c_2 = \beta, c_3 = -\alpha^2, c_1 = 1 - c_2 - c_3 $$
- SSF:
$$SSF_t = c_1 \times \dfrac{C_t + C_{t-1}}{2} + c_2 \times SSF_{t-1} + c_3 \times SSF_{t-2} $$

###### B. Template Code:

In [ ]:
class SuperSmootherFilter(bt.Indicator):

    lines = ('ssf',)
    params = (('period', 10),)

    def __init__(self):
        self.addminperiod(3) # Cần ít nhất nến t, t-1, t-2

        # Tiền tính toán hằng số DSP
        a1 = math.exp(-math.sqrt(2) * math.pi / self.p.period)
        b1 = 2 * a1 * math.cos(math.sqrt(2) * math.pi / self.p.period)

        self.c2 = b1
        self.c3 = -a1 * a1
        self.c1 = 1.0 - self.c2 - self.c3

    def next(self):
        price = self.data.close[0]
        price_1 = self.data.close[-1]

        if len(self) <= 2:
            # Điều kiện biên: Khởi tạo giá trị bằng chính giá
            self.lines.ssf[0] = price
        else:
            # Phương trình đệ quy Butterworth
            ssf_1 = self.lines.ssf[-1]
            ssf_2 = self.lines.ssf[-2]

            self.lines.ssf[0] = self.c1 * ((price + price_1) / 2.0) + self.c2 * ssf_1 + self.c3 * ssf_2

###### C. Ý nghĩa:
- Khi mà thị trường có thanh khoản thấp, giá đi ngang, giật lên 2 điểm rồi sập 2 điểm. Lúc này, SMA và EMA sẽ quấn vào nhau, liên tục báo tín hiệu mua/bán giả. Nhưng SSF thì nằm ngang thành một đường thẳng tắp, trơn tru hoàn hảo.

###### D. Cách dùng:
- Lọc nhiễu: Chỉ vào lệnh khi SSF uốn cong dứt khoát về một hướng.

## 4. Nhóm Thống kê (Statistical - Phân phối)

*Tuyệt đối không tính toán các mô-men thống kê (như Phương sai, Độ lệch, Độ nhọn) trên chuỗi giá OHLCV, là một chuỗi không dừng và chứa nghiệm đơn vị. Nếu tính toán trên giá OHLCV, kết quả sẽ hoàn toàn vô nghĩa về mặt toán học.*

*Do đó, mọi tính toán trong nhóm này sẽ được thực hiện trên Chuỗi Lợi suất (Returns) hoặc Lợi suất Logarit (Log Returns). Nhóm này không dùng để tìm điểm Mua/Bán (Entry/Exit), mà dùng để Định vị Trạng thái thị trường (Regime Identification)*


- Entropy (ENTP): Shannon Entropy. Đo lường mức độ "hỗn loạn" của thị trường. Entropy cao = thị trường ngẫu nhiên, khó đoán. Entropy thấp = có cấu trúc xu hướng rõ ràng.
- Rolling Skew (Độ lệch) & Rolling Kurtosis (Độ nhọn): Phân tích rủi ro đuôi (Tail-risk). Kurtosis cao báo hiệu sắp có biến động mạnh (nổ Vol, nổ biên độ).

**Log Returns**
$$Log\_return_t = ln\Big( \dfrac{C_t}{C_{t-1}} \Big)$$

##### 4.1 Shannon Entropy (ENTP)

###### A. Cơ sở lý thuyết:
- Entropy Thấp: Dòng tiền đang rất đồng thuận. Giá di chuyển theo một cấu trúc rõ ràng, có thể dự đoán được (Trending).
- Entropy Cao: Lực mua và lực bán giằng co ngẫu nhiên, không có phe nào kiểm soát. Thị trường đầy "nhiễu" (White Noise / Choppy).

###### B. Công thức Toán học:

Để tính Entropy, ta phải chuyển đổi chuỗi Lợi suất liên tục thành các Xác suất rời rạc:
- Lấy chuỗi Lợi suất trong chu kỳ $N$ nến. Tìm Max và Min.
- Chia khoảng (Max - Min) thành $k$ rổ bằng nhau.
- Đếm xem có bao nhiêu cây nến có mức lợi suất rơi vào rổ thứ $i$. Gọi tần số này là $F_i$.
- Tính Xác suất để một cây nến bất kỳ rơi vào rổ $i$: $$P_i = \dfrac{F_i}{N}$$
- Công thức Shannon Entropy:
$$ H = -\displaystyle \sum_{i=1}^{k} P_i log_2(P_i) $$

###### C. Template Code:

In [ ]:
import backtrader as bt
import numpy as np

class ShannonEntropy(bt.Indicator):
    lines = ('entropy',)
    params = (('period', 20), ('bins', 10),)

    def __init__(self):
        self.addminperiod(self.p.period + 1)

    def next(self):
        prices = np.array(self.data.close.get(size=self.p.period + 1))

        rets = np.diff(prices) / prices[:-1]

        counts, _ = np.histogram(rets, bins=self.p.bins)

        probs = counts / np.sum(counts)

        probs = probs[probs > 0]

        entropy = -np.sum(probs * np.log2(probs))

        self.lines.entropy[0] = entropy

###### D. Cách dùng:
- Bộ lọc phân loại: Khi nhận thấy Entropy đạt mức cao, tắt chiến lược đánh Trend-following, đồng thời chuyển sang Mean Reversion. Chỉ khi Entropy cắm đầu giảm mạnh mới cho phép mở lệnh Trend-following.

##### 4.2 Rolling Skewness (Độ lệch) & Rolling Kurtosis (Độ nhọn)

###### 4.2.1 Rolling Skewness (Độ lệch - Mô-men bậc 3)
- Cơ sở lý thuyết: Đo lường tính bất đối xứng của phân phối Lợi suất.
- Công thức Toán học:
    + Trung bình mẫu - Kỳ vọng $\mu$): $$ \mu = \dfrac{1}{N} \displaystyle\sum_{j=1}^{N} Log\_return_j $$
    + Độ lệch chuẩn: $$\sigma = \sqrt{ \dfrac{1}{N-1} \displaystyle\sum_{j=1}^{N} (Log\_return_j - \mu)^2 } $$
    + Với $Z_t$ là điểm chuẩn hóa của lợi suất: $$Z_t = \dfrac{Log\_return_t - \mu}{\sigma} $$
    + Skewness:
      $$ Skewness = \dfrac{1}{N} \displaystyle\sum_{t=1}^{N} (Z_t)^3 $$
- Ý nghĩa: Nếu Skewness âm sâu (Lệch trái), nghĩa là số lượng các cây nến giảm điểm cực mạnh nhiều hơn hẳn các cây nến tăng điểm. Nếu đang giữ Long mà thấy Skewness âm dần, đó là cảnh báo dòng tiền lớn đang âm thầm xả hàng, một cú sập gãy sắp diễn ra.

###### 4.2.2 Rolling Excess Kurtosis (Độ nhọn - Mô-men bậc 4)
- Cơ sở lý thuyết: Đo lường độ dày của đuôi phân phối - tức là tần suất xuất hiện các biến động giá cực kỳ lớn (Outliers).
- Công thức Toán học:
      $$ Excess\_Kurtosis = \Bigg[\dfrac{1}{N} \displaystyle\sum_{t=1}^{N} (Z_t)^4\Bigg] - 3 $$
- Ý nghĩa: Khi thị trường biến động thấp, volume giảm. Nhưng nếu lúc này Kurtosis tăng vọt, nó báo hiệu "lò xo đã nén đến cực hạn". Breakout mạn sắp xảy ra.

In [ ]:
import backtrader as bt
import numpy as np

class RollingMoments(bt.Indicator):
    lines = ('skew', 'kurtosis',)
    params = (('period', 20),)

    def __init__(self):
        self.addminperiod(self.p.period + 1)

    def next(self):
        prices = np.array(self.data.close.get(size=self.p.period + 1))
        rets = np.diff(prices) / prices[:-1]

        mean_ret = np.mean(rets)
        std_ret = np.std(rets)

        if std_ret < 1e-8:
            self.lines.skew[0] = 0
            self.lines.kurtosis[0] = 0
            return


        z_scores = (rets - mean_ret) / std_ret

        skewness = np.mean(z_scores ** 3)

        kurt = np.mean(z_scores ** 4) - 3

        self.lines.skew[0] = skewness
        self.lines.kurtosis[0] = kurt

## 5. Nhóm Xu hướng (Trend)
- Choppiness Index (CHOP): Dựa trên hình học phân dạng (Chaos Theory / Fractal Geometry). Xác định xem thị trường đang có xu hướng (Trend) hay đi ngang (Choppy).
- Vortex Indicator (VORTEX): Dựa trên động lực học chất lỏng (Fluid dynamics), đo lường luồng giá xoáy lên và xoáy xuống.
- CORRELATION TREND INDICATOR (CTI): Đo lường sự đồng biến với thời gian

##### 5.1 CHOPPINESS INDEX

###### A. Cơ sở lý thuyết:
- CHOP không quan tâm trend là Tăng hay Giảm. Nó chỉ trả lời một câu hỏi duy nhất: "Thị trường đang đi theo đường thẳng hay đang cuộn tròn?"

###### B. Công thức Toán học:
Ta so sánh "Tổng quãng đường thực tế" với "Độ dời vector lớn nhất".

- Tổng quãng đường thực tế: Là tổng của True Range trong chu kỳ $N$. $$ \displaystyle \sum_{i=1}^N TR_i$$
trong đó: $$TR_t = \max{ \Big( (H_t-L_t), |H_t - C_{t-1}|, |L_t - C_{t-1}| \Big) } $$
- Độ dời vector lớn nhất: Là khoảng cách từ Giá cao nhất đến Giá thấp nhất trong toàn bộ chu kỳ $N$. $$ Max_N(High) - Min_N(Low)$$
- Công thức (Logarit cơ số 10 để chuẩn hóa về thang 0-100):
$$ CHOP = 100 \times \dfrac{ log_{10} \Bigg( \dfrac{ \displaystyle \sum_{i=1}^N TR_i }{Max_N(High) - Min_N(Low)} \Bigg) }{log_{10}(N)} $$

###### C. Template Code:

In [2]:
import backtrader as bt
import math

class ChoppinessIndex(bt.Indicator):
    lines = ('chop',)
    params = (('period', 14),)

    def __init__(self):
        self.addminperiod(self.p.period + 1)
        self.tr = bt.indicators.TrueRange()

    def next(self):
        sum_tr = math.fsum(self.tr.get(size=self.p.period))

        highest_high = max(self.data.high.get(size=self.p.period))
        lowest_low = min(self.data.low.get(size=self.p.period))

        range_hl = highest_high - lowest_low

        if range_hl < 1e-8:
            self.lines.chop[0] = 100
            return

        ratio = sum_tr / range_hl

        if ratio > 0:
            chop = 100 * (math.log10(ratio) / math.log10(self.p.period))
            self.lines.chop[0] = max(0, min(100, chop))
        else:
            self.lines.chop[0] = 100

###### D. Cách dùng:
- $CHOP > 60$: Breakout giả
- $CHOP < 40$: Trend cực mạnh
- Phân loại: Khi $CHOP$ rớt từ vùng 60 xuống dưới 50, đó là lúc "Lò xo bung nén".

##### 5.2 VORTEX

###### A. Cơ sở lý thuyết:
- Positive Vortex: Lực hút từ Đáy hôm qua lên Đỉnh hôm nay. Đại diện cho lực Mua chủ động.
- Negative Vortex: Lực hút từ Đỉnh hôm qua xuống Đáy hôm nay. Đại diện cho lực Bán đè.
Chỉ báo này bóc tách cấu trúc nến cực kỳ sâu sắc, nhìn thấu sự giằng co bên trong thay vì chỉ nhìn vào giá đóng cửa.

###### B. Công thức Toán học:
Tại thời điểm $t$:
- Positive Vortex Movement ($VM^+$): Khởi tạo lực đẩy lên.
$$ VM_t^+ = |H_t - L_{t-1}| $$
- Negative Vortex Movement ($VM^-$): Khởi tạo lực đè xuống.
$$ VM_t^- = |L_t - H_{t-1}| $$
- Chuẩn hóa: Tổng các Vortex Movement trong $N$ chu kỳ chia cho Tổng True Range để chuẩn hóa thành tỷ lệ.
\begin{align*}
VI_N^+ = \dfrac{\displaystyle\sum_{t=1}^N VM_t^+ }{\displaystyle\sum_{t=1}^N TR_t}
\quad \text{ và } \quad
VI_N^- = \dfrac{\displaystyle\sum_{t=1}^N VM_t^- }{\displaystyle\sum_{t=1}^N TR_t} \quad \text{ với } \quad TR_t = \max{ \Big( (H_t-L_t), |H_t - C_{t-1}|, |L_t - C_{t-1}| \Big) }
\end{align*}

###### C. Template Code:

In [ ]:
class VortexIndicator(bt.Indicator):
    lines = ('vi_plus', 'vi_minus',)
    params = (('period', 14),)

    def __init__(self):
        tr = bt.indicators.TrueRange(self.data)
        sum_tr = bt.indicators.SumN(tr, period=self.p.period)

        vm_plus = abs(self.data.high - self.data.low(-1))
        vm_minus = abs(self.data.low - self.data.high(-1))

        sum_vm_plus = bt.indicators.SumN(vm_plus, period=self.p.period)
        sum_vm_minus = bt.indicators.SumN(vm_minus, period=self.p.period)

        self.lines.vi_plus = sum_vm_plus / sum_tr
        self.lines.vi_minus = sum_vm_minus / sum_tr

###### D. Cách dùng:
- Vortex Crossover: Khi $VI^+$ cắt lên trên $VI^-$, đó là lúc "Dòng chảy" từ Đáy hôm qua lên Đỉnh hôm nay đã áp đảo hoàn toàn dòng chảy ngược lại  $\to$ Tín hiệu LONG.
- Khác với MACD bị hội tụ, 2 đường Vortex thường duy trì một khoảng cách rất ổn định khi VN30F1M đang có Trend mượt. Nếu 2 đường VI+ và VI- xoắn chặt vào nhau như sợi dây thừng, đó là lúc thị trường đang ngạt thở (Churning), hãy đứng ngoài quan sát.

##### 5.3 CORRELATION TREND INDICATOR (CTI)

###### A. Cơ sở lý thuyết:
Hệ số tương quan Pearson (Pearson Correlation Coefficient) đo lường mức độ quan hệ tuyến tính giữa 2 biến số. CTI sử dụng lý thuyết này nhưng áp dụng một cách độc đáo:
- Biến $X$ (Độc lập): Chính là Thời gian (Các nến đếm từ $1,2,3,\dots,N$).
- Biêến $Y$ (Phụ thuộc): Là Giá đóng cửa.

CTI sẽ trả về một giá trị từ $-1$ đến $+1$:
- CTI $\approx +1$: Giá và Thời gian có tương quan dương. Cứ mỗi nến trôi qua, giá lại cao hơn nến trước $\to$
 Uptrend tuyệt đối.
- CTI $\approx -1$: Giá và Thời gian có tương quan âm. Thời gian trôi qua, giá cắm đầu xuống đều đặn. $\to$
Downtrend tuyệt đối.
- CTI $\approx 0$: Giá và Thời gian không có bất kỳ mối liên hệ nào. $\to$  Thị trường dao động ngẫu nhiên (Random Walk / Sideway).

###### B. Công thức Toán học:
- Công thức Hệ số Pearson cơ bản:
$$ r = \dfrac{\sum (X_i - \overline{X}) (Y_i - \overline{Y}) }{\sqrt{\sum (X_i - \overline{X})^2 \sum (Y_i - \overline{Y})^2}} $$

Nếu tính toán bằng vòng lặp,  mỗi lần nến chạy ta phải tính lại  $\sum (X_i - \overline{X})^2$. Nhưng vì $X$ luôn là một chuỗi thời gian cố định $[0,1,2,3,\dots,N-1]$, phương sai của nó là một hằng số.

   + Kỳ vọng thời gian: $$\overline{X}=\dfrac{N-1}{2}$$
   + Tổng bình phương độ lệch thời gian: $$\sum (X_i - \overline{X})^2 = \dfrac{N(N^2-1)}{12}$$

###### C. Template Code:

In [ ]:
import backtrader as bt
import numpy as np

class CorrelationTrendIndicator(bt.Indicator):
    lines = ('cti',)
    params = (('period', 14),)

    def __init__(self):
        self.addminperiod(self.p.period)

        N = self.p.period
        self.time_array = np.arange(N)
        self.mean_time = np.mean(self.time_array)

        self.ss_time = (N * (N**2 - 1)) / 12

    def next(self):
        prices = np.array(self.data.close.get(size=self.p.period))
        mean_price = np.mean(prices)

        cov_time_price = np.sum((self.time_array - self.mean_time) * (prices - mean_price))

        ss_price = np.sum((prices - mean_price)**2)

        if ss_price < 1e-8:
            self.lines.cti[0] = 0
            return

        r = cov_time_price / np.sqrt(self.ss_time * ss_price)
        self.lines.cti[0] = max(-1, min(1, r))

###### D. Cách dùng:
- Xác nhận Xu hướng: Trong VN30F1M, nếu CTI vượt qua ngưỡng +0.7 (hoặc giảm dưới -0.7), đây là bằng chứng thống kê cho thấy Dòng tiền đang kiểm soát thị trường một cách có hệ thống. Bất kỳ nhịp giật lùi nào lúc này chỉ là điều chỉnh (Pullback), bạn có thể tự tin đặt lệnh nhồi thuận xu hướng.
- Tín hiệu Phân kỳ Tương quan: Giá tạo đỉnh mới nhưng CTI lại rớt từ +0.8 xuống +0.3 (Mối tương quan với thời gian đang gãy). Đây là tín hiệu xả hàng cực kỳ đáng tin cậy. Động lực đẩy giá đã hết, market maker đang thả trôi thanh khoản.

## 6. Nhóm Độ biến động (Volatility)
- Ulcer Index (UI): Đo lường rủi ro sụt giảm (Downside Risk). Nó tính toán dựa trên các nhịp điều chỉnh từ đỉnh gần nhất. Rất tốt để làm filter quản trị rủi ro.
- Mass Index (MASSI): Dựa trên sự mở rộng biên độ High-Low. Thường dùng để dự báo sự đảo chiều (Reversal) khi Mass Index tăng vọt.
- Normalized Average True Range (NATR): Chuẩn hóa ATR theo phần trăm giá, giúp so sánh Volatility giữa các cổ phiếu có thị giá khác nhau một cách công bằng.

##### 6.1 ULCER INDEX (UI)

###### A. Cơ sở lý thuyết:
- Ulcer Index là một thước đo Rủi ro Rút vốn (Drawdown Risk). Nó trừng phạt dựa trên 2 yếu tố: Độ sâu của cú sập và Thời gian kẹt hàng.
- Chỉ báo này cực tốt để làm Filter Quản trị Rủi ro.

###### B. Công thức Toán học:
Hàm số của Ulcer Index sử dụng phép tính Trung bình bình phương (Root Mean Square - RMS). Việc bình phương có tác dụng: Cú sập càng sâu, mức phạt càng bị khuếch đại theo cấp số nhân.

- Tại thời điểm $t$, tìm mức giá đóng cửa cao nhất trong $N$ phiên gần nhất:
$$Max_N(C_t) = \max{(C_{t-N+1},\dots,C_t)}$$
- Tính phần trăm sụt giảm (Drawdown) tại $t$:
$$D_t = 100 \times \dfrac{C_t - Max_N(C_t)}{Max_N(C_t)}$$
($D_t$ luôn $\leq 0$. Nếu giá tạo đỉnh mới, $D_t=0$)
- Tính Trung bình bình phương của chuỗi $D_t$ trong chu kỳ $N$:
$$UI_t = \sqrt{\dfrac{1}{N} \displaystyle\sum_{i=0}^{N-1} (D_{t-i})^2 }$$

###### C. Template Code:

In [ ]:
import backtrader as bt
import numpy as np
import math

class UlcerIndex(bt.Indicator):
    lines = ('ulcer',)
    params = (('period', 14),)

    def __init__(self):
        self.addminperiod(self.p.period)
        self.highest_close = bt.indicators.Highest(self.data.close, period=self.p.period)

    def next(self):
        closes = np.array(self.data.close.get(size=self.p.period))
        max_closes = np.array(self.highest_close.get(size=self.p.period))

        max_closes[max_closes == 0] = 1e-8

        drawdowns = 100 * (closes - max_closes) / max_closes

        squared_dd = drawdowns ** 2
        mean_squared = np.mean(squared_dd)

        self.lines.ulcer[0] = math.sqrt(mean_squared)

###### D. Cách dùng:
- Tín hiệu xác nhận Xu hướng Giảm: Khi UI bắt đầu ngóc đầu tăng mạnh từ mức 0, điều đó có nghĩa là giá liên tục tạo đáy mới và rời xa đỉnh $\to$ Dòng tiền đang xả hàng dứt khoát.

##### 6.2 MASS INDEX (MASSI)

###### A. Cơ sở lý thuyết:
- Khi một xu hướng chuẩn bị đảo chiều, các nhà giao dịch sẽ hoảng loạn hoặc giằng co mạnh nhất, dẫn đến việc Biên độ nến (High - Low) bị nở rộng ra một cách bất thường.
Mass Index không dùng giá Close, nó chỉ tập trung vào True Range của nến.
- Hiện tượng "Reversal Bulge" (Bướu đảo chiều): Khi MASSI vượt lên trên ngưỡng 27, sau đó gãy xuống dưới 26.5 $\to$ Xác suất 80% thị trường chuẩn bị quay xe đảo chiều.

###### B. Công thức Toán học:
- Tính: $$EMA_1 = EMA(High - Low,9)$$
- Tính: $$EMA_2 = EMA(EMA_1,9)$$
- Tính Tỷ lệ Mở rộng (Expansion Ratio): $$Ratio = \dfrac{EMA_1}{EMA_2}$$
- Tính tổng tích lũy trong $N$ phiên:
$$MASSI = \displaystyle\sum_{i=1}^N Ratio_i$$

###### C. Template Code:

In [ ]:
class MassIndex(bt.Indicator):
    """
    Mức quan trọng: Vượt 27 (Cảnh báo), rớt lại 26.5 (Kích hoạt đảo chiều).
    """
    lines = ('massi',)
    params = (
        ('ema_period', 9),
        ('sum_period', 25),
    )

    def __init__(self):
        hl_range = self.data.high - self.data.low

        ema1 = bt.indicators.EMA(hl_range, period=self.p.ema_period)

        ema2 = bt.indicators.EMA(ema1, period=self.p.ema_period)

        ratio = bt.If(ema2 > 0, ema1 / ema2, 1.0)

        self.lines.massi = bt.indicators.SumN(ratio, period=self.p.sum_period)

###### D. Cách dùng:
- Bắt đỉnh/đáy.

##### 6.3 NORMALIZED AVERAGE TRUE RANGE (NATR)

###### A. Cơ sở lý thuyết:
- NATR (Normalized ATR) chuẩn hóa ATR về đơn vị phần trăm (%).

###### B. Công thức Toán học:
- Tính True Range (TR) có xét đến Gap mở cửa:
$$ TR_t =\max{(H_t-L_t, |H_t - C_{t-1}|, |L_t - C_{t-1}|)} $$
- Làm mượt bằng Trung bình (thường dùng Wilder's Moving Average hoặc Simple Moving Average): $$ATR_t = SMA(TR_t,N)$$ (thông thường $N=14$)
- Chuẩn hóa: $$NATR_t = 100\times \dfrac{ATR_t}{C_t} $$

###### C. Template Code:

In [ ]:
class NormalizedATR(bt.Indicator):
    lines = ('natr',)
    params = (('period', 14),)

    def __init__(self):
        self.atr = bt.indicators.ATR(self.data, period=self.p.period)

    def next(self):
        close_price = self.data.close[0]
        if close_price > 1e-8:
            self.lines.natr[0] = 100 * (self.atr[0] / close_price)
        else:
            self.lines.natr[0] = 0

## 7. Nhóm Khối lượng (Volume)
- Ease of Movement (EOM): Sự kết hợp giữa động lượng giá và khối lượng. EOM cao tức là giá tăng rất dễ dàng mà không cần nhiều Volume (cạn cung).
- Elder's Force Index (EFI): Volume * (Close_t - Close_{t-1}). Cực kỳ đơn giản nhưng mạnh mẽ để đo áp lực mua/bán thực tế.
- Price Volume Rank (PVR):

##### 7.1 EASE OF MOVEMENT (EOM)

###### A. Cơ sở lý thuyết:
- Richard Arms phát triển EOM dựa trên triết lý: "Thị trường đi lên dễ dàng nhất khi không có lực cản (Cạn cung), và rơi tự do khi không có lực đỡ (Trống cầu)."
    + EOM Cao (Dương): Giá tăng rất mạnh, nhưng Volume lại rất nhỏ. Điều này chứng tỏ phe Bán đã kiệt sức, phe Mua chỉ cần một lực rất nhẹ cũng đẩy giá lên cao.
    + EOM Thấp (Âm): Giá rơi mạnh với Volume nhỏ. Phe Mua đã buông xuôi.
    + EOM xoay quanh mốc 0: Giá có thể tăng/giảm nhẹ nhưng Volume lại khổng lồ. Đây là dấu hiệu của sự Hấp thụ. Hai phe đang đánh nhau vỡ đầu tại một vùng giá, báo hiệu sắp có nổ xu hướng.

###### B. Công thức Toán học:
EOM được thiết kế như một hàm Tỷ lệ nghịch giữa "Quãng đường dịch chuyển" và "Mật độ Khối lượng".
- Quãng đường dịch chuyển (Midpoint Move):
$$MM_t = \dfrac{H_t+L_t}{2} - \dfrac{H_{t-1}+L_{t-1}}{2} $$
- Mật độ Khối lượng (Box Ratio):
$$ Box\_Ratio = \dfrac{ V_t }{H_t - L_t} $$
- Công thức 1-Period EOM:
$$EOM_t = \dfrac{MM_t}{Box\_Ratio_t} $$
- Thường được làm mượt bằng $SMA(EOM,14)$

###### C. Template Code:

In [ ]:
import backtrader as bt

class EaseOfMovement(bt.Indicator):
    lines = ('eom',)
    params = (('period', 14), ('scale', 10000.0), ('epsilon', 1e-8),)

    def __init__(self):
        midpoint = (self.data.high + self.data.low) / 2
        midpoint_move = midpoint - midpoint(-1)

        hl_range = self.data.high - self.data.low
        box_ratio = (self.data.volume / self.p.scale) / (hl_range + self.p.epsilon)

        eom_1 = midpoint_move / (box_ratio + self.p.epsilon)

        self.lines.eom = bt.indicators.SMA(eom_1, period=self.p.period)

###### D. Cách dùng:
- Xác nhận Breakout (Phá vỡ): Nếu Giá vượt kháng cự kèm theo EOM tăng vọt, đó là một cú Breakout uy tín.
- Phát hiện bẫy (Bull Trap): Giá vượt đỉnh nhưng EOM lại cắm đầu đi xuống mốc 0. Điều này giải mã ra là: Để nhích lên được 1 điểm, đội Long phải tốn một lượng Volume khổng lồ do đội Short liên tục chặn trên $\to$ Nỗ lực (Volume) không mang lại Kết quả (Price). Sắp sập!

##### 7.2 ELDER'S FORCE INDEX (EFI) - "Định luật 2 Newton trong Trading"

###### A. Cơ sở lý thuyết:
- Tiến sĩ Alexander Elder thiết kế EFI dựa trên nền tảng của Vật lý học (Định luật 2 Newton: $F = ma$ (Lực bằng Khối lượng nhân Gia tốc).
- Trong thị trường:
    + Khối lượng (Mass): Là Volume giao dịch.
    + Gia tốc (Acceleration): Là mức thay đổi của Giá Đóng cửa ($C_t - C_{t-1}$).
- EFI cho chúng ta biết Áp lực thực tế (True Pressure) đằng sau một cây nến. Một cây nến xanh tăng 5 điểm với Volume 10,000 sẽ có "Lực" mạnh gấp 5 lần một cây nến xanh tăng 5 điểm nhưng chỉ có Volume 2,000.

###### B. Công thức Toán học:
- Công thức thô:
$$Force_t = Volume_t \times (C_t - C_{t-1}) $$
- Do $Force_t$ biến động rất nhiễu, ta bắt buộc phải đưa nó qua một bộ lọc thông thấp (thường là đường EMA chu kỳ 13).
$$EFI_{13} = EMA(Force_t,13)$$

###### C. Template Code:

In [ ]:
class EldersForceIndex(bt.Indicator):
    lines = ('efi',)
    params = (('period', 13),)

    def __init__(self):
        price_change = self.data.close - self.data.close(-1)

        raw_force = self.data.volume * price_change
        self.lines.efi = bt.indicators.EMA(raw_force, period=self.p.period)

###### D. Cách dùng:
- Phân kỳ Động lượng (Force Divergence): Bắt đỉnh/đáy. Khi thị trường rớt xuống tạo đáy mới sâu hơn, nhưng EFI lại tạo đáy cao hơn. Nó chứng minh rằng: "Giá bị ép xuống, nhưng Volume đã cạn kiệt". Lập tức đảo chiều LONG.

##### 7.3 PRICE VOLUME RANK (PVR)

###### A. Cơ sở lý thuyết:
- PVR là một thuật toán chấm điểm sự kết hợp giữa Giá và Volume, được chia làm 4 trạng thái của thị trường:
    1. Giá Tăng + Volume cao: Bò kiểm soát mạnh mẽ (Long mạnh).
    2. Giá Tăng + Volume thấp: Short Squeeze hoặc Tăng ảo (Dễ bẫy).
    3. Giá Giảm + Volume thấp: Pullback lành mạnh (Chỉnh để lên tiếp).
    4. Giá Giảm + Volume cao: Gấu kiểm soát mạnh mẽ (Short mạnh).

###### B. Công thức Toán học:
- Dấu của Lợi suất (Sign of Return):
$$Sign_t \begin{cases} 1 \quad &\text{ nếu } C_t > C_{t-1} \\ -1 \quad &\text{ nếu } C_t < C_{t-1} \\ 0 \quad &\text{ nếu } C_t = C_{t-1} \end{cases}$$
- Hạng phân vị Volume (Volume Percentile Rank): Xếp hạng Volume hiện tại nằm ở vị trí % thứ mấy trong $N$ phiên gần nhất.
$$ VolRank_t = 100 \times \dfrac{ \text{Số phiên có Volume } < V_t }{N} $$
- Công thức PVR:
$$PVR_t = Sign_t \times VolRank_t$$

###### C. Template Code:

In [ ]:
import numpy as np

class PriceVolumeRank(bt.Indicator):
    """
    Price Volume Rank (PVR): Chuẩn hóa Hành vi Khối lượng - Giá thành một thang điểm duy nhất [-100, 100].
    +100: Nến tăng có khối lượng lớn nhất trong chu kỳ.
    -100: Nến sập có khối lượng lớn nhất trong chu kỳ.
    """
    lines = ('pvr',)
    params = (('period', 20),)

    def __init__(self):
        self.addminperiod(self.p.period)

    def next(self):
        delta_price = self.data.close[0] - self.data.close[-1]
        sign = 1 if delta_price > 0 else (-1 if delta_price < 0 else 0)

        vol_array = np.array(self.data.volume.get(size=self.p.period))
        current_vol = vol_array[-1]

        count_less = np.sum(vol_array < current_vol)

        vol_rank = 100 * (count_less / float(self.p.period - 1)) if self.p.period > 1 else 50

        self.lines.pvr[0] = sign * vol_rank

###### D. Cách dùng:
- Trạng thái Cạn kiệt: Cây nến đóng cửa đỏ (giảm), nhưng $PVR = -5$ (tức là dấu âm, nhưng Volume nằm ở top bét bảng 5% thấp nhất) $\to$ Đây là cú rũ hàng không có thanh khoản, không được SHORT dù cho giá thủng hỗ trợ".